# 🎯 Python: Generators & Lazy Evaluation

**Advanced Python Concepts for Interview Preparation**

- **Created:** 2026-02-28
- **Focus:** `yield`, Lazy Evaluation, Memory-Efficient Streaming
- **Tags:** `python` | `generators` | `memory-optimization` | `interview-prep`

## 📖 The Core Concept in Plain Language

A standard function calculates everything, loads it all into your computer's RAM, and `return`s the entire massive chunk at once — then destroys its local state.

A **Generator** uses the `yield` keyword instead. When Python hits `yield`, it spits out a single value and hits **"pause"**. The function's state — variables, loop counters, where it was — is perfectly frozen. When you ask it for the next value (using `next()`), it unpauses, runs until the next `yield`, and freezes again.

This is called **Lazy Evaluation**: computing data only exactly when it is requested.

### Why This Matters

- **Memory:** A list materializes everything into RAM at once. A generator processes *one element at a time*, giving you $O(1)$ memory regardless of dataset size.
- **Performance:** Don't pay the cost of computing elements you never use.
- **Scale:** The same code that handles 1,000 rows handles 1,000,000,000 rows — no changes required.

### The Key Insight

```python
# LIST: loads ALL 1 million numbers into RAM immediately
numbers = [x * 2 for x in range(1_000_000)]   # ~8 MB in RAM

# GENERATOR: computes one number at a time, on demand
numbers = (x * 2 for x in range(1_000_000))   # ~200 bytes in RAM
```

## 🔍 Generators as First-Class Objects: The Basics

In [ ]:
import sys

# ✅ 1. The yield keyword — pauses, returns a value, resumes
def count_up(n):
    print("Generator started")
    for i in range(n):
        print(f"  About to yield {i}")
        yield i
        print(f"  Resumed after yielding {i}")
    print("Generator done")

gen = count_up(3)
print("Generator object created — nothing executed yet!")
print(type(gen))  # <class 'generator'>
print()

# Each call to next() advances to the next yield
print("Calling next() #1:", next(gen))
print()
print("Calling next() #2:", next(gen))
print()
print("Calling next() #3:", next(gen))

print()

# ✅ 2. Memory comparison — list vs generator
N = 1_000_000
list_mem   = sys.getsizeof(list(range(N)))
gen_mem    = sys.getsizeof(x for x in range(N))
print(f"List size:      {list_mem:,} bytes  ({list_mem // 1024 // 1024} MB)")
print(f"Generator size: {gen_mem} bytes")
print(f"Memory ratio:   {list_mem // gen_mem:,}x more RAM for the list")

## 🏗️ The Anatomy of a Generator

A generator is a function that uses `yield` instead of `return`. Python automatically transforms it into a **suspended state machine**.

### Structure:
```python
def generator_function(input_data):    # ← Regular function signature
    for item in input_data:            # ← Iterate over source
        processed = transform(item)    # ← Do work on ONE item
        yield processed               # ← Pause & return this value
                                      #    State is frozen here
    # When iteration ends, StopIteration is raised automatically
```

In [ ]:
# Full anatomy: a generator that parses transaction records
def parse_transactions(raw_lines):
    """Yields one parsed transaction dict at a time — never loads all into RAM."""
    for line in raw_lines:
        line = line.strip()
        if not line or line.startswith('#'):
            continue  # Skip blank lines and comments
        parts = line.split(',')
        yield {
            'id':     parts[0],
            'amount': float(parts[1]),
            'type':   parts[2]
        }


# Simulated raw file (in production, this would be open('file.csv'))
raw = [
    "# Transaction log",
    "TXN001,150.00,CREDIT",
    "TXN002,-45.50,DEBIT",
    "",
    "TXN003,2000.00,CREDIT",
]

# Iterate: each transaction is processed one at a time
for tx in parse_transactions(raw):
    print(f"  Processed: {tx}")

## 🔄 Generator Expressions vs List Comprehensions

A **Generator Expression** is syntactic sugar — the lazy version of a list comprehension.

Same syntax, one character difference: `[]` (list) vs `()` (generator).

In [ ]:
import sys

data = range(100_000)

# Approach 1: List Comprehension — materializes everything NOW
squares_list = [x ** 2 for x in data]
print(f"List:      {sys.getsizeof(squares_list):,} bytes — ALL values in RAM")

# Approach 2: Generator Expression — computes on demand
squares_gen = (x ** 2 for x in data)
print(f"Generator: {sys.getsizeof(squares_gen)} bytes — no values computed yet")

print()

# Both work identically in for loops
total_gen  = sum(x ** 2 for x in range(10))   # Generator: O(1) memory
total_list = sum([x ** 2 for x in range(10)])  # List: O(n) memory
print(f"Both produce same result: {total_gen} == {total_list}: {total_gen == total_list}")

print()

# ✅ PREFER generator form for large sum/max/min/any/all operations
huge_data = range(10_000_000)
result = sum(x for x in huge_data if x % 2 == 0)  # No list ever created
print(f"Sum of even numbers 0-10M: {result:,}")

## 🌍 Real-World Example: Streaming Large File Processing

**The Citi Context:** Thousands of transaction files, each multiple gigabytes. Engineers were loading entire files into Python lists — causing `OutOfMemoryError` crashes in production.

### Without Generators (Crashes at Scale)
```python
def process_transactions(filepath):
    with open(filepath) as f:
        all_lines = f.readlines()        # 💥 Loads entire 10 GB file into RAM
    records = [parse(line) for line in all_lines]  # 💥 Second copy in RAM
    return [r for r in records if r['amount'] > 1000]  # 💥 Third pass
```

### With Generators (Constant Memory, Any File Size)
```python
@stream_processor
def process_transactions(filepath):
    with open(filepath) as f:
        yield from (parse(line) for line in f)  # One line at a time
```

In [ ]:
import io

# Simulate a multi-GB file with a StringIO stream
def make_fake_file(n_rows):
    """Creates a fake CSV in memory to simulate a large file."""
    lines = ["txn_id,amount,currency,status"]
    for i in range(n_rows):
        lines.append(f"T{i:06d},{(i % 5000) + 0.99:.2f},USD,{'SETTLED' if i % 3 else 'PENDING'}")
    return io.StringIO('\n'.join(lines))


# ✅ Generator-based pipeline — processes row by row
def read_transactions(file_obj):
    """Yields one raw row at a time from the file."""
    next(file_obj)  # Skip header
    for line in file_obj:
        yield line.strip()

def parse_transactions(raw_rows):
    """Parses each raw row into a structured dict."""
    for row in raw_rows:
        parts = row.split(',')
        yield {'id': parts[0], 'amount': float(parts[1]),
               'currency': parts[2], 'status': parts[3]}

def filter_large(transactions, threshold=1000.0):
    """Passes through only high-value transactions."""
    for tx in transactions:
        if tx['amount'] >= threshold:
            yield tx


# Build pipeline — nothing executes until we iterate
fake_file    = make_fake_file(n_rows=50_000)
raw_rows     = read_transactions(fake_file)
parsed       = parse_transactions(raw_rows)
high_value   = filter_large(parsed, threshold=4000.0)

# Drive the pipeline — processes 50K rows with O(1) memory
results = list(high_value)
print(f"Processed 50,000 rows — found {len(results)} high-value transactions")
print(f"Sample: {results[0]}")

## 🎭 Advanced: Infinite Generators & `yield from`

In [ ]:
import itertools

# ✅ Infinite generator — no upper bound needed
def fibonacci():
    """Infinite Fibonacci sequence — runs forever, uses O(1) memory."""
    a, b = 0, 1
    while True:       # No stopping condition!
        yield a
        a, b = b, a + b

# Take only what you need (without storing the rest)
first_10 = list(itertools.islice(fibonacci(), 10))
print(f"First 10 Fibonacci: {first_10}")

first_large = next(x for x in fibonacci() if x > 1000)
print(f"First Fibonacci > 1000: {first_large}")

print()

# ✅ yield from — delegate to another generator (flatten)
def chunked_file_reader(files):
    """Reads multiple files as a single seamless stream."""
    for f in files:
        yield from f   # Delegates to each file's iterator

# Simulate two file streams
file_1 = iter(["line1-file1", "line2-file1"])
file_2 = iter(["line1-file2", "line2-file2"])
stream = chunked_file_reader([file_1, file_2])

print("Streaming two files as one:")
for line in stream:
    print(f"  {line}")

print()

# ✅ Generator cannot be indexed or reversed
gen = (x for x in range(5))
try:
    print(gen[2])   # This will fail
except TypeError as e:
    print(f"Cannot index a generator: {e}")
    print("Generators are forward-only state machines.")

## 🎤 5 Interview Q&A

### Q1: What is the difference between `yield` and `return`?

**Answer:** `return` sends a value back to its caller and *destroys* the function's local state. `yield` produces a value and *suspends* the function's execution, preserving local variables so it can resume exactly where it left off on the next `next()` call. A function with `yield` becomes a generator function that returns a generator object — nothing in it executes until you iterate.

---

### Q2: Why would you use a Generator instead of a List?

**Answer:** Memory efficiency. A list materializes every element into RAM at once — $O(n)$ space. A generator processes one element at a time (Lazy Evaluation), giving you $O(1)$ memory regardless of dataset size. This allows you to process infinitely large datasets — like millions of transaction log lines or a 10 GB CSV — with a tiny, constant memory footprint.

---

### Q3: How do you manually step through a generator?

**Answer:** You use the built-in `next()` function: `val = next(my_gen)`. Each call advances the generator to its next `yield`. When it runs out of items, it raises `StopIteration`. In practice, you almost never call `next()` manually — a `for` loop does it for you automatically.

---

### Q4: What is a Generator Expression?

**Answer:** A high-performance, memory-efficient shortcut to create a generator in one line. It looks exactly like a List Comprehension but uses parentheses `()` instead of brackets `[]`.
```python
my_gen = (x * 2 for x in huge_dataset)   # Generator — O(1) memory
my_list = [x * 2 for x in huge_dataset]  # List — O(n) memory
```
Prefer generator expressions when feeding into `sum()`, `max()`, `min()`, `any()`, `all()` — they never need the full list.

---

### Q5: Can you reverse or index into a generator? (e.g., `gen[5]`)

**Answer:** No. Generators are **forward-only state machines**. They do not hold data in memory, so they have no concept of an "index" or "length". You can only ask for the `next()` item. If you need random access, materialize into a list first — but that defeats the memory benefit.

## 📚 Key Terminology to Master

| Term | Definition | Example |
|------|-----------|----------|
| **Lazy Evaluation** | Computing data only when requested, not upfront | Generator yields one item at a time |
| **Eager Evaluation** | Computing everything immediately and storing it | List comprehension: `[x for x in data]` |
| **Suspended State Machine** | A generator's internal state (variables, position) is frozen between `yield` calls | Generator pauses at `yield`, resumes with `next()` |
| **Streaming vs Materializing** | Streaming = process one-at-a-time; Materializing = load all into RAM | Generator vs list |
| **Generator Expression** | One-line generator using `()` syntax | `(x**2 for x in data)` |
| **`yield from`** | Delegates iteration to another iterable (flattens one level) | `yield from sub_generator` |
| **`StopIteration`** | Exception raised when a generator is exhausted | Caught automatically by `for` loops |
| **$O(1)$ Space** | Constant memory footprint regardless of input size | Generator uses same memory for 100 or 100M rows |

## 💼 The Citi Narrative: From Crashes to Constant Memory

### The Problem

At Citi, we process massive files of **millions of transactions every single day**. Early on, pipelines crashed constantly from `OutOfMemoryError` because engineers would try to load an entire 10-Gigabyte XML transaction file into a standard Python list to parse it. A single file would consume 30+ GB of RAM — crashing the container and the downstream jobs depending on it.

### The Solution

I mandated using **Python generators** (or PySpark partitions, which use the same lazy evaluation concept under the hood) to stream the files. Our generators `yield` one parsed transaction at a time, keeping the container's RAM footprint virtually zero — regardless of whether the file is 10 MB or 100 GB.

### The Code Pattern

```python
def stream_transactions(filepath):
    """O(1) memory — handles any file size."""
    with open(filepath, 'r') as f:
        next(f)  # skip header
        for line in f:                  # Python's file iterator is lazy
            yield parse_transaction(line)  # one transaction at a time

# Pipeline: each step is a generator — nothing materializes
raw      = stream_transactions('transactions_20260228.csv')  # 10 GB file
filtered = (tx for tx in raw if tx.amount > THRESHOLD)
enriched = (enrich(tx) for tx in filtered)

for tx in enriched:    # Only here does data flow
    write_to_db(tx)
```

### The Impact

- **Reliability:** Zero `OutOfMemoryError` crashes after the change
- **Scalability:** Same pipeline handles 10 MB or 100 GB files without code changes
- **Performance:** No wasted computation on records that get filtered out
- **Pattern Transfer:** PySpark partitions are essentially distributed generators — same mental model scales to petabyte workloads

## 💪 Practice Exercises

In [ ]:
# EXERCISE 1: Write a generator that yields only even numbers from a list
# Rules: use yield, not list comprehension, handle any iterable input

def even_numbers(data):
    for item in data:
        if item % 2 == 0:
            yield item

result = list(even_numbers(range(1, 21)))
print(f"Exercise 1: {result}")
# Expected: [2, 4, 6, 8, 10, 12, 14, 16, 18, 20]

In [ ]:
# EXERCISE 2: Write a generator pipeline that:
# 1. Reads raw strings like "Alice,85" (name,score)
# 2. Parses them into dicts
# 3. Filters to scores >= 80
# All as separate generators — no intermediate lists

def parse_scores(raw_lines):
    for line in raw_lines:
        name, score = line.split(',')
        yield {'name': name, 'score': int(score)}

def filter_passing(records, threshold=80):
    for record in records:
        if record['score'] >= threshold:
            yield record

raw = ["Alice,85", "Bob,72", "Charlie,91", "Diana,78", "Eve,88"]
parsed   = parse_scores(raw)
passing  = filter_passing(parsed)

print("Exercise 2 — passing students:")
for student in passing:
    print(f"  {student}")
# Expected: Alice(85), Charlie(91), Eve(88)

In [ ]:
# EXERCISE 3: Write a generator that produces a running cumulative sum
# For input [1, 2, 3, 4, 5] it should yield 1, 3, 6, 10, 15
# Bonus: make it work on infinite sequences (don't load all into memory)

def running_total(numbers):
    total = 0
    for n in numbers:
        total += n
        yield total

result = list(running_total([1, 2, 3, 4, 5]))
print(f"Exercise 3a: {result}")
# Expected: [1, 3, 6, 10, 15]

# Bonus: works on infinite sequence too
import itertools
first_5_natural_running = list(itertools.islice(running_total(itertools.count(1)), 5))
print(f"Exercise 3b: {first_5_natural_running}")
# Expected: [1, 3, 6, 10, 15]

## 🎯 Summary: Key Takeaways

### What You've Learned

1. ✅ **`yield` vs `return`:** `yield` pauses and preserves state; `return` terminates and destroys state
2. ✅ **Lazy Evaluation:** Generators compute one item at a time — $O(1)$ memory regardless of dataset size
3. ✅ **Generator Expressions:** `()` instead of `[]` — same syntax, lazy execution
4. ✅ **Pipeline Pattern:** Chain generators together — no intermediate lists, no wasted memory
5. ✅ **`yield from`:** Delegate to sub-generators; flatten nested iterables cleanly
6. ✅ **Infinite Sequences:** Generators can produce values forever — only compute what you consume
7. ✅ **Forward-only:** Generators cannot be indexed or reversed — they're one-way state machines

### When to Use Generators

- ✅ **DO** use generators when processing large files or datasets line-by-line
- ✅ **DO** use generator expressions instead of lists in `sum()`, `max()`, `any()`, `all()`
- ✅ **DO** use generator pipelines to avoid intermediate materializations
- ❌ **DON'T** use generators when you need random access (indexing) or multiple passes
- ❌ **DON'T** convert to a list immediately — that defeats the whole purpose

### Interview Confidence Checklist

- [ ] Can explain `yield` vs `return` with precision
- [ ] Can write a generator function from scratch
- [ ] Can write a generator expression vs list comprehension
- [ ] Can chain generators into a pipeline without intermediate lists
- [ ] Can explain why `gen[2]` fails — and what to do instead
- [ ] Can name 3+ real-world use cases (file streaming, infinite sequences, pipelines)
- [ ] Can tie generators to PySpark's lazy evaluation model

---

**"Simplicity and clarity is Gold."** — Sean's Study Mantra

Master generators and you've mastered one of Python's most powerful patterns for building production-grade, memory-efficient data pipelines. 🚀